# Set-up part 1: Run this cell first

This cell loads the Python libraries required (numpy, scipy and plotting routines). It also provides the routine sheplog to define Toft's modified version of the Shepps-Logan phantom image. This image is a standard image in testing the mathematics of tomography. Toft's modification makes it easier to appreciate the process, as the original Shepps-Login image is designed to stress the mathematics. Here, we are primarily interested in the basic principles, so we use Toft's modified version.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy

# this code enables the automated feedback. If you remove this, you won't get any feedback
# so don't delete this cell!
try:
  import AutoFeedback
except (ModuleNotFoundError, ImportError):
  %pip install AutoFeedback
  import AutoFeedback

#try:
#  from testsrc import test_main
#except (ModuleNotFoundError, ImportError):
#  %pip install "git+https://github.com/autofeedback-exercises/exercises.git#subdirectory=MTH2021/Radon"
#  from testsrc import test_main

#try:
#  from testsrc.test_main import sheplog
#except (ModuleNotFoundError, ImportError):
#  print("whoops")
#  pass

def runtest(tlist):
  import unittest
  from contextlib import redirect_stderr
  from os import devnull
  with redirect_stderr(open(devnull, 'w')):
    suite = unittest.TestSuite()
    for tname in tlist:
      suite.addTest(eval(f"test_main.UnitTests.{tname}"))
    runner = unittest.TextTestRunner()
    try:
      runner.run(suite)
    except AssertionError:
      pass

def sheplog(N):
    #
    # Toft's modified Shepp-Logan picture
    #
    #
    # It is created as a list of ellipses
    # Each entry in Elli describes a separate ellipse
    # Parameters: center y, center x
    #             minor r, major r
    #             angle w.r.t. y, shading value
    #
    # original shading values:
    # 2 -.98 -0.02 -0.02 remainder 0.01
    #
    # modified:
    # 1 -.8 -.2 -.2 others .1
    #
    Elli=[]
    Elli.append((0,0,0.69,0.92,0,1))
    Elli.append((0,-0.0184,0.6624,0.874,0,-0.8))
    Elli.append((0.22,0,0.11,0.31,-18,-0.2))
    Elli.append((-0.22,0,0.16,0.41,18,-0.2))
    Elli.append((0,0.35,0.21,0.25,0,0.1))
    Elli.append((0,0.1,0.046,0.046,0,0.1))
    Elli.append((0,-0.1,0.046,0.046,0,0.1))
    Elli.append((-0.08,-0.605,0.046,0.023,0,0.1))
    Elli.append((0,-0.605,0.023,0.023,0,0.1))
    Elli.append((0.06,-0.605,0.023,0.046,0,0.1))
    slimg=np.zeros((N,N))
    xcoord = np.linspace(1,-1,N)
    ycoord = np.linspace(-1,1,N)
    for component in Elli:
        ycent=component[0]
        xcent=component[1]
        xlen=component[3]
        ylen=component[2]
        angle= component[4]
        gray = component[5]
        for j in range(N):
            for k in range(N):
                x=(xcoord[j]-xcent)*np.cos(angle/180*np.pi)-(ycoord[k]-ycent)*np.sin(angle/180*np.pi)
                y=(ycoord[k]-ycent)*np.cos(angle/180*np.pi)+(xcoord[j]-xcent)*np.sin(angle/180*np.pi)
                if ((((x)/xlen)**2+((y)/ylen)**2)<=1.):
                    slimg[j,k] += gray
#    plt.imshow(slimg,cmap = 'gray', interpolation='none', aspect = 'equal')
#    plt.colorbar()
#    plt.show()
    return slimg


# Setup part 2: Run this cell next: define the Shepp-Logan phantom image


The code to define the starting image is written for you. The picture is known as Toft's modified Shepp-Logan phantom, and the function to make the picture is called `sheplog`. You need to build it by running the code below. The code will also plot it to see what you're aiming for.

The code requires a parameter N to identify the size of the image. By default it is set to 256.

In [ ]:

N= 256 # This defines the dimension of the image: 256X256 pixels

phantom = sheplog(N)
plt.imshow(phantom, cmap='gray', interpolation='none', aspect='equal', extent=[-1, 1, -1, 1])
plt.colorbar()
plt.xlabel('horizontal displacement')
plt.ylabel('vertical displacement')
plt.show()

# Setup - part 3 : Run this cell next - Define the spatial and angular grid

We also need to build arrays for the coordinates that we will use in the rest of our calculations. Namely

* `grid`: holding `N` linearly spaced grid-values between -1 and 1 (effectively).
* `angles`: holding `M` linearly spaced angle-values between 0 and $\dfrac{(M-1)\pi}{M}$

This code requires a parameter $M$ for the number of angles at which images are taken. The default value is set to 30.

The final calculation takes a bit of time. Test your code initially for this default value. Only change it when you are certain the code is running correctly.

Note, we do not include 180 degrees (or pi radians) in our list of angles, as this angle provides the same data as 0 degrees. The Fourier transforms and reconstruction will not work properly if this angle is double counted.

In [ ]:
M = 180
dtheta = np.pi / M
grid_spacing = 2/N

grid = np.linspace(-1+1/N, 1-1/N, N)
angles = np.linspace(0, np.pi*(M-1)/M, M)

# Set-up part 4: Run this cell next - Apply the Radon Transform to the Shepp-Logan phantom

The Radon transform represents the actual outcome obtained by the CT scanner. To build this image from our Shepp-Logan image, run the cell below.

The Radon transform image is stored in the variable 'Radon_image' with dimensions $(N,M)$.

In [ ]:
Radon_image = np.zeros((N,M))

fit_grid = np.zeros((N,2))

for m, angle in enumerate(angles):
    for n, displacement in enumerate(grid):
       fit_grid[:, 0] = np.linspace(-np.cos(angle), np.cos(angle), N)-displacement * np.sin(angle)
       fit_grid[:, 1] = np.linspace(-np.sin(angle), np.sin(angle), N)+displacement * np.cos(angle)
       in_data = scipy.interpolate.interpn((grid, grid), phantom, fit_grid,  method='linear', bounds_error=False, fill_value=0)
       Radon_image[n, m]=sum(in_data) * 2. / N

plt.imshow(Radon_image,interpolation='none',aspect='auto', extent=[0, 174, -1,1])
plt.xlabel('Angle (°)')
plt.ylabel('Perpendicular displacement')
plt.show()

# Your work - Step 1:
# Define the wavenumber grid associated with the (inverse) Radon Transform

Both numpy and scipy have Fourier transform packages "fft". It is helpful to read the documentation for these routines.

One particularly useful auxiliary routine is the "fftfreq" routine. This provides the wavenumbers associated with the output of the Fourier transform routine fft. These wavenumbers are essential for reconstruction of the original image.

An advantage of the "fftfreq" routine is that it provides wavenumbers in the format we need. The wavenumber grid starts at 0, and increases by $1/L$, where $L$ is the overall length of the spatial grid.

In a discrete Fourier transformation, $N/L$ and $0$ are equivalent wavenumbers. Thus, wavenumbers of $(N-1)/L$ and $-1/L$ are equivalent. We need the wavenumbers with least magnitude. The "fftfreq" will ensure that fft data is provided on a grid associated with wavenumbers of least magnitude.

We should define two auxiliary variables derived from these wavenumbers:
- abs_k, giving the absolute magnitude of the wavenumbers
- phase_k, giving a (potential) phase shift that needs to be applied to the output of the Fourier transform.

The magnitude is needed for the transformation from a cylindrical to a Cartesian wavenumber grid.

The phase transform is needed, because we define the image on a grid from $[-1,1]$, whereas the Fourier transform routine assumes the data is given on an interval $[0,2]$.

In the CT scan, we rotate around the centre of the image, and it is thus easier to have the origin at the centre of the image.

In [ ]:
k_grid= np.fft.fftfreq(N, d=2/N)         # Use the np.fft.fftfreq routine to store the wavenumber grid in the variable k_grid

abs_k = np.abs(k_grid)      # Assign the magnitude of the wavenumber to the variable abs_k
phase_k = np.exp(-2j*np.pi*k_grid)        # Assign the phase change required for each wavenumber due to the interval change to phase_k

# Your work - step 2:
# Determine the Fourier transform of the Radon image along the displacement coordinate

Use the "fft" routine from the scipy package to carry out a Fourier transform of the Radon image. The Fourier transform must be taken along the displacement coordinate for each angle at which a Radon transform is obtained.

Adjust the coefficient for the phase factor and for the weight factor determined earlier.  

This Fourier transform will need normalisation. Multiply the entire array by the spatial step-size of $2/N$. In addition, the Fourier transform in the python packages does not include our standard pre-factor of $1/\sqrt{2\pi}$.

In [ ]:
Radon_Fourier = np.zeros((N,M), dtype = complex)

for m in range(M):                     # create a loop over all angles
    Radon_Fourier[:, m] = np.fft.fft(Radon_image[:, m])  # Use np.fft.fft, abs_k and phase_k to get the Fourier transform for each angle
    Radon_Fourier[:, m] = Radon_Fourier[:, m] * phase_k * abs_k   # including the adjustment of output coefficients for phase change
                                   # the weight factor associated with k

Radon_Fourier = Radon_Fourier * (2/N)     # renormalise coefficients (see text)

# Your work - step 3.
# Define a 2D grid for the restored image

We need to introduce some variables for the reconstruction. The reconstruction is based upon a superposition of terms $e^{i 2\pi (k_x x +k_y y)}$. These terms are complex, but we are only interested in the real components. We use two arrays within the reconstruction: the individual terms will be stored in an intermediate complex array, whereas the final image will be stored in a real array.

Secondly, we can simplify the reconstruction if we generate a variable that contains the $x$ and $y$ coordinates in a 2D arrays containing the coordinates for all points of interest. We can obtain this 2D grid in a variable grid_2d by using the numpy "meshgrid" function and our 'grid' array of displacement coordinates. Using "meshgrid", the variable grid_2d should contain 2 sets of 2D coordinates: grid_2d[0] contains the x coordinate, and grid_2d[1] the y-coordinate.


In [ ]:
Recon_comp=np.zeros((N,N),dtype=complex)
Recon=np.zeros((N,N))

grid_2d = np.meshgrid(grid, grid)                    # Obtain 2D grids for x and y coordinates using np.meshgrid

# Your work: step 4
# Reconstruct the image

Now we have all the data required to reconstruct our initial image as a superposition of inverse Fourier transforms including all angles.

We need a loop over all angles.

For each angle:
- We have Fourier transform data associated with wavenumbers perpendicular to this angle, but we need these wavenumbers in Cartesian form: $k_x$ and $k_y$. We thus need to use the angle to transform the wavenumbers accordingly.
- For each wavenumber pair $(k_x,k_y)$, we need to determine the inverse 2D Fourier transform $Ce^{i 2\pi (k_x x +k_y y)}$, where $C$ is the corresponding coefficient in the Fourier expansion of the Radon transform. Calculate this inverse transform and store it in Recon_comp. Then add the real part of Recon_comp to the reconstructed image in the variable Recon.

Renormalise the final image by multiplying it by $1/(N\pi)$.


In [ ]:
for m, angle in enumerate(angles):                                       # loop over angles
    print(f'evaluating angle number {m}', end='\r')   # This displays progress of the calculation -- it takes a bit of time
    kx = k_grid * np.cos(angle)                    # Project the wavenumber grid onto the k_x coordinate for the specific angle
    ky = k_grid * np.sin(angle)                       # Project the wavenumber grid onto the k_y coordinate for the specific angle
    for n in range(N):                                   # Loop over the wavenumber grid
        Recon_comp = Radon_Fourier[n, m] * np.exp(
            2j*np.pi*(kx[n]*grid_2d[0] + ky[n]*grid_2d[1])
        )                                                # Construct the 2D Fourier component for this specific wavenumber and angle combination
        Recon += np.real(Recon_comp)                     # Add the real part of this component to the reconstruction of the final image

Recon = Recon / (N*np.pi)                                 # Renormalise the image


# Your work - step 5:
# Show the final image (no coding needed)

Note: if the image appears mirrored or rotated compared to the original, reconsider your definition of $k_x$ and $k_y$.

If no image is being reproduced, carefully check your work. Mathematics can be very powerful, but a correct image is only returned when every single part is completed correctly. Any mistake will lead to an incorrect image.

In [ ]:
plt.imshow(Recon, interpolation='none', cmap='gray',extent=[-1,1,-1,1])
plt.xlabel('horizontal displacement')
plt.ylabel('vertical displacement')
plt.colorbar()
plt.show()

# Your work - step 6

# Tidying the figure - no coding needed

The Radon transform was obtained by rotating a $[-1$ to $1, -1$ to $1]$ image around the origin. Hence, only a circle with radius 1 around the origin will be captured fully when all angles are considered. Anything outside this circle will not captured only partially, and the information shown cannot be considered reliable.

To show the reliable part of the image, we first set all data outside the circle to a central value. Once that is done, we can safely rescale the reliable part of the figure on a 0 - 1 scale, and set all data outside the circle to 0.


In [ ]:
innerdatapoint = Recon[int(N/2),int(N/2)]
Recon = Recon*( (grid_2d[1]**2+grid_2d[0]**2)<=1.) + \
    innerdatapoint*( (grid_2d[1]**2+grid_2d[0]**2)>1.)

Reconmx = np.max(Recon)
Reconmin = np.min(Recon)

Recon = (Recon-Reconmin)/(Reconmx-Reconmin) \
    *(np.sqrt(grid_2d[1]**2+grid_2d[0]**2)<=1.) + \
    0. *( (grid_2d[1]**2+grid_2d[0]**2)>1.)

plt.imshow(Recon,interpolation='none',cmap='gray', extent=[-1, 1, -1, 1])
plt.xlabel('horizontal displacement')
plt.ylabel('vertical displacement')
plt.colorbar()
plt.show()